# Chapter 37 — The Agent Core on Its Own

**Companion to *Pi Agents*** · [`Pi Agents` chapter 37](https://programmer.ie/books/pi/37-chapter/)

| | |
|---|---|
| Chapter | `37-chapter.md` · weight 37 · `/books/pi/37-chapter/` |
| Notebook | `notebooks/pi/37-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does one prompt produce a four-message transcript and exactly two provider
requests — and is a lookup miss data or a failure?

The chapter's point: `pi-agent-core` is the loop with **no coding agent, no host,
no terminal**. A tool-using prompt produces a four-message transcript
(user → assistant-with-call → toolResult → assistant-answer), exactly one
provider request per turn, and events in order. A lookup that finds nothing is
**data, not a failure**.


## What this notebook establishes

- One tool-using prompt produces a **four-message transcript**: user, assistant (tool call), toolResult, assistant (answer).
- Two turns cost **exactly two provider requests** — one per turn.
- Events arrive in order: `tool_execution_start` before `tool_execution_end`, and the run ends with `agent_end`.
- A tool that finds nothing returns **data, not a failure** — `isError: false` with `no matches`.

## What this notebook does **not** establish

- **The absence of `agent_settled` from the core's types is a reading of declarations** (`ch37-lim3`), not a runtime observation here.
- **'What the host was providing' is the book's summary** (`ch37-lim2`) — the core runs alone by design.
- **The model is scripted** — request counts are exact for the scripted control flow only.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + Inspect (declarations) — the chapter's research agent against the scripted provider, plus a static count of the core's imports.
* **Evidence scope:** `in_process_runtime` + `declarations`.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(37))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/37-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the loop, in four messages and two requests

In [3]:
CORE = pinb.driver_source("ch37-agent-core.ts")
core = pinb.driver(CORE, label="ch37-agent-core", timeout=180)
l = core["loop"]
print("Role sequence:", " -> ".join(l["roles"]))
print(pinb.md_table(
    ["check", "value"],
    [["request count", l["callCount"]],
     ["tool events in order", l["toolOrder"]],
     ["last event", l["lastEvent"]]],
))

Role sequence: user -> assistant -> toolResult -> assistant
| check | value |
|---|---|
| request count | 2 |
| tool events in order | True |
| last event | agent_end |


In [4]:
pinb.show_checks([
    pinb.check("the transcript has the four-message shape",
               l["roles"] == ["user", "assistant", "toolResult", "assistant"], " -> ".join(l["roles"])),
    pinb.check("two turns cost exactly two requests",
               l["callCount"] == 2, f"{l['callCount']} requests"),
    pinb.check("tool_execution_start precedes tool_execution_end",
               l["toolOrder"], "ordered"),
    pinb.check("the run ends with agent_end",
               l["lastEvent"] == "agent_end", "agent_end"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  --------------------------------------------
PASS  the transcript has the four-message shape  user -> assistant -> toolResult -> assistant
PASS  two turns cost exactly two requests  2 requests
PASS  tool_execution_start precedes tool_execution_end  ordered
PASS  the run ends with agent_end  agent_end

4/4 assertions held.


## A lookup miss is data, not a failure

In [5]:
m = core["miss"]
print(pinb.md_table(
    ["check", "value"],
    [["isError", m["isError"]],
     ["content", m["content"]]],
))

| check | value |
|---|---|
| isError | False |
| content | [{"type":"text","text":"no matches"}] |


In [6]:
pinb.show_checks([
    pinb.check("a lookup that finds nothing is not a failure",
               m["isError"] is False, "isError false"),
    pinb.check("it returns the data the tool gives: no matches",
               "no matches" in m["content"], "content"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------  -------------
PASS  a lookup that finds nothing is not a failure  isError false
PASS  it returns the data the tool gives: no matches  content

2/2 assertions held.


## The chapter's own tests

Both agent-core tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch37-agent-core/research-agent.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch37-agent-core/research-agent.test.ts', show="loop")

$ node --test ch37-agent-core/research-agent.test.ts
  PASS  2 passed, 0 failed, 0 skipped  in 448 ms

  tests matching 'loop':
    ok   the loop runs tool then answer, and the transcript records both


## Your turn: predict, then run

**Predict first.** The chapter's research agent runs `lookup_evidence` then answers.
What would the role sequence be if the model called the tool **twice** before
answering?

**Then change one input.** The corpus has two entries. Add a third that also
matches the query. Does the tool return more than one entry — and does the
four-message shape change?

**Predict the boundary.** The core's event stream ends with `agent_end`. Where does
`agent_settled` come from — the core, or the host?

In [8]:
print("Predictions:")
print("  1. Two tool calls: user, assistant, toolResult, toolResult, assistant - five messages.")
print("  2. More matches: the tool returns what it has; the shape is unchanged.")
print("  3. agent_settled is a host concept; the core ends with agent_end.")
print()
print("Observed above:")
print(f"  roles: {" -> ".join(l['roles'])}")
print(f"  requests: {l['callCount']}, tool order ok: {l['toolOrder']}, last: {l['lastEvent']}")
print(f"  miss: isError={m['isError']}, content={m['content']}")
print()
assert l["roles"] == ["user", "assistant", "toolResult", "assistant"]
assert l["callCount"] == 2 and l["toolOrder"] and l["lastEvent"] == "agent_end"
assert m["isError"] is False and "no matches" in m["content"]
print("held: the core loops tool-then-answer; a miss is data; events are ordered")

Predictions:
  1. Two tool calls: user, assistant, toolResult, toolResult, assistant - five messages.
  2. More matches: the tool returns what it has; the shape is unchanged.
  3. agent_settled is a host concept; the core ends with agent_end.

Observed above:
  roles: user -> assistant -> toolResult -> assistant
  requests: 2, tool order ok: True, last: agent_end
  miss: isError=False, content=[{"type":"text","text":"no matches"}]

held: the core loops tool-then-answer; a miss is data; events are ordered


## Interpretation

The agent core alone is the loop, with the boundaries the chapter shows:

| Claim | Evidence |
|---|---|
| **Four-message transcript** | user → assistant (tool call) → toolResult → assistant |
| **One request per turn** | Two turns, `callCount == 2` |
| **Events in order** | `tool_execution_start` < `tool_execution_end`, ends with `agent_end` |
| **A miss is data** | `isError: false`, content `no matches` |

The practical rules:
1. **The core is host-free** — no terminal, no context loader, no permissions. The host supplies those.
2. **Count requests to find loop bugs** — a turn should cost one request.
3. **A tool result is data** — read `isError` and the content, not the shape.
4. **`agent_end` is the core's terminal event** — `agent_settled` belongs to the host.

## Sources, execution mode and limitations

**Execution mode:** **Run** + Inspect (declarations) — the chapter's research agent against the scripted provider, plus a static count of the core's imports.

**Evidence scope:** `in_process_runtime` + `declarations`.

### What was read or run

- **Ran** `drivers/ch37-agent-core.ts`: the chapter's research agent twice (tool-using loop, lookup miss).
- **Ran** `ch37-agent-core/research-agent.test.ts` (2 tests).
- **Read** `examples/evidence.json`, chapter 37 rows (2 claims).

### Limitations

- **The absence of `agent_settled` from the core's types is a reading of declarations** (`ch37-lim3`).
- **The model is scripted** — request counts are exact for the scripted control flow only.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
